In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, broadcast, sum, avg, count, round, max,
    row_number, rank, lag,
    to_date, year, month, quarter, datediff, current_date,
    months_between, dayofweek, when,
    split, explode, size, array_contains,
    regexp_replace, coalesce, date_format
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("SalesAnalyticsEngine") \
    .master("local[*]") \
    .getOrCreate()

# ──────────────────────────────────────────────
# TABLE 1: Orders (the big fact table)
# ──────────────────────────────────────────────
orders_data = [
    (1001, 1, 101, "2024-01-15", 2, 29.99),
    (1002, 2, 102, "2024-01-20", 1, 899.99),
    (1003, 3, 101, "2024-01-28", 3, 15.99),
    (1004, 1, 103, "2024-02-03", 1, 1200.00),
    (1005, 4, 101, "2024-02-14", 2, 49.99),
    (1006, 2, 104, "2024-02-18", 1, 350.00),
    (1007, 5, 102, "2024-03-01", 4, 15.99),
    (1008, 3, 103, "2024-03-08", 1, 450.00),
    (1009, 1, 101, "2024-03-15", 2, 29.99),
    (1010, 6, 104, "2024-03-22", 1, 89.99),
    (1011, 4, 102, "2024-04-01", 1, 1200.00),
    (1012, 2, 101, "2024-04-10", 3, 49.99),
    (1013, 5, 103, "2024-04-15", 2, 29.99),
    (1014, 1, 104, "2024-04-22", 1, 899.99),
    (1015, 3, 101, "2024-05-01", 2, 350.00),
    (1016, 6, 102, "2024-05-10", 1, 49.99),
    (1017, 4, 103, "2024-05-18", 3, 15.99),
    (1018, 2, 104, "2024-05-25", 1, 450.00),
    (1019, 1, 101, "2024-06-05", 2, 89.99),
    (1020, 5, 102, "2024-06-15", 1, 1200.00),
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date_str", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
])

df_orders = spark.createDataFrame(orders_data, orders_schema)

# ──────────────────────────────────────────────
# TABLE 2: Products (small lookup — has categories as comma-separated string)
# ──────────────────────────────────────────────
products_data = [
    (1, "Gaming Laptop",   800.00, 0.25, "electronics,gaming,computers"),
    (2, "Office Chair",    200.00, 0.40, "furniture,office"),
    (3, "Mechanical KB",    80.00, 0.35, "electronics,gaming,accessories"),
    (4, "Standing Desk",   400.00, 0.30, "furniture,office,ergonomic"),
    (5, "Webcam",           45.00, 0.50, "electronics,office,accessories"),
    (6, "Monitor",         300.00, 0.20, "electronics,computers"),
]

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("cost", DoubleType(), False),        # What we pay to supplier
    StructField("margin_pct", DoubleType(), False),  # Profit margin percentage
    StructField("categories", StringType(), False),  # Comma-separated!
])

df_products = spark.createDataFrame(products_data, products_schema)

# ──────────────────────────────────────────────
# TABLE 3: Customers (small lookup)
# ──────────────────────────────────────────────
customers_data = [
    (101, "Alice",   "Kampala"),
    (102, "Bob",     "Nairobi"),
    (103, "Charlie", "Kigali"),
    (104, "Diana",   "Kampala"),
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("city", StringType(), False),
])

df_customers = spark.createDataFrame(customers_data, customers_schema)

# ──────────────────────────────────────────────
# TABLE 4: Regions (tiny lookup — maps cities to regions)
# ──────────────────────────────────────────────
regions_data = [
    ("Kampala",  "East Africa",  "Uganda"),
    ("Nairobi",  "East Africa",  "Kenya"),
    ("Kigali",   "East Africa",  "Rwanda"),
    ("Lagos",    "West Africa",  "Nigeria"),   # No customers here
]

regions_schema = StructType([
    StructField("city", StringType(), False),
    StructField("region", StringType(), False),
    StructField("country", StringType(), False),
])

df_regions = spark.createDataFrame(regions_data, regions_schema)

In [3]:
df_orders_dated = (
    df_orders
    .withColumn("order_date", to_date(col("order_date_str"), "yyyy-MM-dd"))
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", date_format("order_date", "MMMM"))  # Full month name
    .withColumn("order_quarter", quarter("order_date"))  # keep quarter separately
    .withColumn(
        "day_type",
        when(dayofweek("order_date").isin(1, 7), "weekend")
        .otherwise("weekday")
    )
    .drop("order_date_str")
)
null_dates = df_orders_dated.filter(col("order_date").isNull()).count()
print(f"Failed date parses: {null_dates}")   # Should be 0

df_orders_dated.show(5)

Failed date parses: 0
+--------+----------+-----------+--------+----------+----------+----------+-----------+-------------+--------+
|order_id|product_id|customer_id|quantity|unit_price|order_date|order_year|order_month|order_quarter|day_type|
+--------+----------+-----------+--------+----------+----------+----------+-----------+-------------+--------+
|    1001|         1|        101|       2|     29.99|2024-01-15|      2024|    January|            1| weekday|
|    1002|         2|        102|       1|    899.99|2024-01-20|      2024|    January|            1| weekend|
|    1003|         3|        101|       3|     15.99|2024-01-28|      2024|    January|            1| weekend|
|    1004|         1|        103|       1|    1200.0|2024-02-03|      2024|   February|            1| weekend|
|    1005|         4|        101|       2|     49.99|2024-02-14|      2024|   February|            1| weekday|
+--------+----------+-----------+--------+----------+----------+----------+-----------+---

In [4]:
df_orders_enriched = df_orders_dated.withColumn(
    "line_revenue",
    round(col("quantity")*col("unit_price"),2)
)

df_orders_enriched.select(
    "order_id","quantity","unit_price","line_revenue"
).show(5)

+--------+--------+----------+------------+
|order_id|quantity|unit_price|line_revenue|
+--------+--------+----------+------------+
|    1001|       2|     29.99|       59.98|
|    1002|       1|    899.99|      899.99|
|    1003|       3|     15.99|       47.97|
|    1004|       1|    1200.0|      1200.0|
|    1005|       2|     49.99|       99.98|
+--------+--------+----------+------------+
only showing top 5 rows



In [5]:
# ──────────────────────────────────────────────
# STEP 3: Join all tables — broadcast the small ones
# ──────────────────────────────────────────────

# Join 1: Orders + Products (broadcast products — only 6 rows)
df_joined = df_orders_enriched.join(
    broadcast(df_products),
    on="product_id",
    how="inner"
)
df_joined= df_orders_enriched.join(
    broadcast(df_products),
    on = "product_id",
    how= "inner"
)

# Join 2: + Customers (broadcast customers — only 4 rows)
df_joined = df_joined.join(
    broadcast(df_customers),
    on="customer_id",
    how="inner"
)

# Join 3: + Regions (broadcast regions — only 4 rows)
df_joined = df_joined.join(
    broadcast(df_regions),
    on="city",
    how="left"   # Left join in case a city has no region mapping
)

# Verify: check that we didn't lose rows
print(f"Orders before join: {df_orders.count()}")
print(f"Orders after join:  {df_joined.count()}")

# Check the join strategy
df_joined.explain()

Orders before join: 20
Orders after join:  20
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [city#24, customer_id#2, product_id#1, order_id#0, quantity#4, unit_price#5, order_date#34, order_year#42, order_month#51, order_quarter#61, day_type#72, line_revenue#150, product_name#13, cost#14, margin_pct#15, categories#16, customer_name#23, region#29, country#30]
   +- BroadcastHashJoin [city#24], [city#28], LeftOuter, BuildRight, false
      :- Project [customer_id#2, product_id#1, order_id#0, quantity#4, unit_price#5, order_date#34, order_year#42, order_month#51, order_quarter#61, day_type#72, line_revenue#150, product_name#13, cost#14, margin_pct#15, categories#16, customer_name#23, city#24]
      :  +- BroadcastHashJoin [customer_id#2], [customer_id#22], Inner, BuildRight, false
      :     :- Project [product_id#1, order_id#0, customer_id#2, quantity#4, unit_price#5, order_date#34, order_year#42, order_month#51, order_quarter#61, day_type#72, line_revenue#150, prod

In [10]:
# Add profit metrics (now that we have product cost info from the join)
df_full = df_joined.withColumn(
    "line_cost",
    round(col("quantity")*col("cost"), 2)
).withColumn(
    "line_profit",
    round(col("line_revenue")-col("line_cost"),2)
).withColumn(
    "profit_margin",
    round((col("line_revenue") - col("line_cost")) / col("line_revenue") * 100, 1)
)

# Inspect the enriched data
df_full.select(
    "order_id", "product_name", "customer_name", "region", "quantity","cost",
    "line_revenue", "line_cost", "line_profit", "profit_margin"
).show(5, truncate=False)

+--------+-------------+-------------+-----------+--------+-----+------------+---------+-----------+-------------+
|order_id|product_name |customer_name|region     |quantity|cost |line_revenue|line_cost|line_profit|profit_margin|
+--------+-------------+-------------+-----------+--------+-----+------------+---------+-----------+-------------+
|1001    |Gaming Laptop|Alice        |East Africa|2       |800.0|59.98       |1600.0   |-1540.02   |-2567.6      |
|1002    |Office Chair |Bob          |East Africa|1       |200.0|899.99      |200.0    |699.99     |77.8         |
|1003    |Mechanical KB|Alice        |East Africa|3       |80.0 |47.97       |240.0    |-192.03    |-400.3       |
|1004    |Gaming Laptop|Charlie      |East Africa|1       |800.0|1200.0      |800.0    |400.0      |33.3         |
|1005    |Standing Desk|Alice        |East Africa|2       |400.0|99.98       |800.0    |-700.02    |-700.2       |
+--------+-------------+-------------+-----------+--------+-----+------------+--

In [11]:
# ──────────────────────────────────────────────
# STEP 4A: Revenue by region and month
# ──────────────────────────────────────────────
df_regional_monthly=(
    df_full
    .groupBy("region","country","order_month")
    .agg(
        round(sum("line_revenue"),2).alias("total_revenue"),
        round(avg("line_revenue"),2).alias("avg_order_value"),
        count("order_id").alias("total_orders"),
        round(sum("line_profit"),2).alias("total_profit")
    )
    .orderBy("region","order_month")
)

df_regional_monthly.show(truncate=False)

+-----------+-------+-----------+-------------+---------------+------------+------------+
|region     |country|order_month|total_revenue|avg_order_value|total_orders|total_profit|
+-----------+-------+-----------+-------------+---------------+------------+------------+
|East Africa|Kenya  |April      |1200.0       |1200.0         |1           |800.0       |
|East Africa|Rwanda |April      |59.98        |59.98          |1           |-30.02      |
|East Africa|Uganda |April      |1049.96      |524.98         |2           |-350.04     |
|East Africa|Rwanda |February   |1200.0       |1200.0         |1           |400.0       |
|East Africa|Uganda |February   |449.98       |224.99         |2           |-550.02     |
|East Africa|Uganda |January    |107.95       |53.97          |2           |-1732.05    |
|East Africa|Kenya  |January    |899.99       |899.99         |1           |699.99      |
|East Africa|Uganda |June       |179.98       |179.98         |1           |-1420.02    |
|East Afri

In [12]:
# ──────────────────────────────────────────────
# STEP 4B: Weekday vs Weekend
# ──────────────────────────────────────────────
df_day_type = (
    df_full
    .groupBy("day_type")
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("line_revenue"), 2).alias("total_revenue"),
        round(avg("line_revenue"), 2).alias("avg_order_value")
    )
)

df_day_type.show()

+--------+------------+-------------+---------------+
|day_type|total_orders|total_revenue|avg_order_value|
+--------+------------+-------------+---------------+
| weekday|          13|       4063.8|          312.6|
| weekend|           7|      4195.93|         599.42|
+--------+------------+-------------+---------------+

